In [ ]:
import numpy as np
import datetime
import os
from netCDF4 import Dataset
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import itertools
np.float = float
import cartopy
from regrid import regrid
import tqdm
import warnings
warnings.filterwarnings("ignore")

In [ ]:
datapath_aux = '/Users/carmennab/Dropbox/pb_new/data/roughness/auxiliary/'
ease_lons = np.load(f'{datapath_aux}1.1km_lon_cent.npy')
ease_lats = np.load(f'{datapath_aux}1.1km_lat_cent.npy')

In [ ]:
### mask out data outside of study area
datapath_aux = '/Users/carmennab/Dropbox/pb_new/auxiliary/'
region_mask = np.array(Dataset(f'{datapath_aux}NSIDC_Regions_Mask_ease.nc')['mask'])
mask_lat = np.array(Dataset(f'{datapath_aux}NSIDC_Regions_Mask_ease.nc')['latitude'])
mask_lon = np.array(Dataset(f'{datapath_aux}NSIDC_Regions_Mask_ease.nc')['longitude'])
mask_ease = regrid(region_mask,mask_lon,mask_lat,ease_lons,ease_lats)
mask_ease[ease_lats<55] = 0
mask_ease[ease_lats>62] = 0
mask_ease[ease_lons>-76] = 0

In [ ]:
d1 = datetime.date(2010, 1, 1)
d2 = datetime.date(2022, 12, 31)
dates = [d1 + datetime.timedelta(days=x) for x in range((d2-d1).days + 1)]
str_dates = [date.strftime('%Y%m%d') for date in dates]

In [ ]:
datapath_in = '/Users/carmennab/Dropbox/pb_new/data/SIT/CS2SMOS/'
for date,str_date in zip(dates,str_dates):
    if str_date[4:6] == '04':
        filepath = f'{datapath_in}{date.year}/{str_date}.nc'
        if os.path.exists(filepath) == True: 
            file = Dataset(filepath)
            lons = np.array(file['longitude'])
            lats = np.array(file['latitude'])
            sit = np.array(file['sea_ice_thickness'])
            sd = np.array(file['snow_depth'])

            sit_ease = regrid(sit,lons,lats,ease_lons,ease_lats)
            sd_ease = regrid(sd,lons,lats,ease_lons,ease_lats)

            sit_ease[~np.isin(mask_ease,4)] = np.nan # remove non-HB data
            sd_ease[~np.isin(mask_ease,4)] = np.nan # remove non-HB data
            
            if (~np.isnan(sit_ease).all()) or (~np.isnan(sd_ease).all()):
                filepath_out = f'{datapath_in}/1.1km/{str_date}.nc'
                if os.path.exists(filepath_out) == True: 
                    ! rm {filepath_out}
                data_vars = {'sea_ice_thickness':(['x','y'],sit_ease),
                             'snow_depth':(['x','y'],sd_ease)}
                ds = xr.Dataset(data_vars = data_vars,
                                coords={'longitude':(['x','y'],ease_lons),
                                        'latitude':(['x','y'],ease_lats)})
                ds.to_netcdf(filepath_out,'w')

In [ ]:
datapath_in = '/Users/carmennab/Dropbox/pb_new/data/SIC/'
for date,str_date in zip(dates,str_dates):
    if str_date[4:6] == '04':
        filepath = f'{datapath_in}{date.year}/{str_date}.nc'
        if os.path.exists(filepath) == True: 
            file = Dataset(filepath)
            lons = np.array(file['longitude'])
            lats = np.array(file['latitude'])
            sic = np.array(file['sic'])
            
            sic_ease = regrid(sic,lons,lats,ease_lons,ease_lats)
            sic_ease[~np.isin(mask_ease,4)] = np.nan # remove non-HB data
            
            if (~np.isnan(sic_ease).all()):
                filepath_out = f'{datapath_in}/1.1km/{str_date}.nc'
                if os.path.exists(filepath_out) == True: 
                    ! rm {filepath_out}
                data_vars = {'sic':(['x','y'],sic_ease)}
                ds = xr.Dataset(data_vars = data_vars,
                                coords={'longitude':(['x','y'],ease_lons),
                                        'latitude':(['x','y'],ease_lats)})
                ds.to_netcdf(filepath_out,'w')